#project


For Groq Install Libraries

In [ ]:
!pip install langchain wikipedia -q
!pip install -U langchain-groq
!pip install requests

For groq import Libraries

In [ ]:
from langchain_groq import ChatGroq
import requests
from getpass import getpass

GROQ_API_KEY = getpass("Enter API Key: ")

Setup Groq

In [ ]:
client = ChatGroq(
   api_key=GROQ_API_KEY,
   model="llama-3.3-70b-versatile",
   temperature=0
)
print(GROQ_API_KEY)

From Here OpenAI 

For OpenAI Install Libraries

In [ ]:
!pip install langchain-openai

For OpenAI import Libraries

In [ ]:
from langchain_openai import ChatOpenAI
from getpass import getpass

OPENAI_API_KEY = getpass("Enter API Key: ")

Setup OpenAI client

In [ ]:
client = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model="gpt-4.1",
    temperature=0
)
print(OPENAI_API_KEY)


In [ ]:

response = client.invoke("Hello")
print(response.content)

Code

In [16]:
HEADERS = {
    "User-Agent": "Note Generator"
}

def get_wiki_details(topic):
    r= requests.get("https://en.wikipedia.org/w/api.php",
             params={"action": "query","titles": topic,
                   "prop": "extracts","exintro":True,
                   "explaintext":True,"format":"json"},
             headers=HEADERS, timeout=10)
    pages = r.json()["query"]["pages"]
    page=next(iter(pages.values()))
    return page.get("extract","no details found.")[:2000]

In [ ]:
BLOCKED_WORDS=["bomb","weapon","hack","kill","malware"]

def is_safe(text):
    return not any(word in text.lower()for word in BLOCKED_WORDS)


def agent(role, instruction, message):
    response = client.invoke([
        {
            "role": "user",
            "content": f"{instruction}\n\nInput: {message}"
        }
    ]).content
    print(f"\n[{role}]\n{response}")
    return response


goal = " Write a short note on India."


if not is_safe(goal):
    print("Blocked:unsafe request.")
else:
    search_term = agent(
        "PLANNER",
        "What is the main topic in the goal? Reply with ONLY one word. No explanation.",
        goal
    ).strip().strip('"').strip("'").split()[0]

    print(f"Search term extracted: '{search_term}'")

    info = get_wiki_details(search_term)
    print(f"\n[Tool] get_wiki_details('{search_term}')\n{info[:300]}")

    facts = agent(
        "RESEARCHER",
        "Extract exactly 5 key facts as bullet points from this text. Only bullet points, nothing else.",
        info[:1500]
    )

    draft = agent(
        "WRITER",
        "Using only these facts, write 5 simple sentences a school student can understand.",
        f"Topic: {search_term}\nFacts:\n{facts}"
    )

    score = agent(
        "EVALUATOR",
        "Rate these notes 1-10. First line only 1 number. Second line one sentence feedback. Nothing else.",
        f"Topic: {search_term}\nDraft:\n{draft}"
    )


try:
    if int(score.splitlines()[0]) < 7:
        draft = agent(
            "WRITER(RETRY)",
            "Improve these notes using the feedback. Keep it 5 simple sentences.",
            f"Draft:\n{draft}\n\nFeedback:\n{score}"
        )

except ValueError:
    pass


print("\n" + "=" *50)
print("Final Study Notes:\n", draft)

